# DL Foundations: Convolutional Neural Networks in CV. Part 2.

# Популярные архитектуры


!pip install torchsummary timm resnest

In [1]:
import torch
from torch import nn
from torchvision.models import vgg16
from torchsummary import summary
import timm
import torchvision.models as models

# Для ResNeSt нужен отдельный пакет: pip install resnest
from resnest.torch import resnest50

## 1. VGG (2014, Simonyan & Zisserman)
![vgg.png](assets/vgg.png)
- **Идея:** глубокие сети с повторяющимися блоками `Conv3x3 + ReLU`.  
- **Особенности:**
  - Однотипные блоки → легко анализировать.
  - Очень много параметров (VGG-16: 138M).
  - Проблема глубины: vanishing gradients.
- **Интересная практика обучения:**
  1. Сначала обучали **мелкие варианты сети** (VGG-A/B/C), чтобы убедиться, что архитектура стабильно обучается.  
  2. Потом постепенно увеличивали глубину до полной (VGG-16/VGG-19), проверяя стабильность градиентов.  
- **Архитектура:**
  - Блок: `[Conv3x3 → ReLU] x N → MaxPool2x2`
  - 16–19 слоев + 3 FC → Softmax
- **Плюсы:** простая и предсказуемая структура, легко масштабируется.  
- **Минусы:** большой размер модели, медленный inference.


In [2]:

def get_summary(model: nn.Module, full_summary: bool = False) -> None:
    # Статистика по обучаемым параметрам
    model.eval()
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Всего параметров: {total_params:,}")
    print(f"Обучаемых параметров: {trainable_params:,}")
    # Подробная сводка по слоям
    if full_summary:
        summary(model, (3, 224, 224), device="cpu")

vgg_model = vgg16()
get_summary(vgg_model, full_summary=True)

Всего параметров: 138,357,544
Обучаемых параметров: 138,357,544
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            Conv2d-8        [-1, 128, 112, 112]         147,584
              ReLU-9        [-1, 128, 112, 112]               0
        MaxPool2d-10          [-1, 128, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]         295,168
             ReLU-12          [-1, 256, 56, 56]               0
           Conv2d-13          [-1, 256,

## 2. ResNet (2015, He et al.)
![resnet_nn](assets/resnet.png)


- **Идея:** **Residual / Skip connections** — градиент легко проходит через блоки.
  
![ResNet-residual-block](assets/ResNet-residual-block.png)
ResNet (2015, He et al.)

- **Residual Block:**
$$
y = F(x) + x
$$

### Типы блоков ResNet
$Conv^* = (Conv + BN + ReLU)$
| Блок | Слои | Применение | Особенности |
|------|------|------------|-------------|
| **BasicBlock** | $3×3\ Conv^* → 3×3\ Conv^* →+ SkipConn$ | ResNet-18 / 34 | простой residual блок для относительно неглубоких сетей |
| **Bottleneck** | $1×1\ Conv^* → 3×3\ Conv^*  → 1×1\ Conv^*  →+ SkipConn$ | ResNet-50 / 101 / 152 | уменьшение размерности внутри блока|

- **Плюсы:** глубокие сети (>100 слоев) без vanishing gradients
- **Минусы:** больше вычислений и памяти по сравнению с VGG

In [3]:
from torchvision.models import resnet50
resnet_model = resnet50()
get_summary(resnet_model)

Всего параметров: 25,557,032
Обучаемых параметров: 25,557,032


## 3. DenseNet (2017, Huang et al.)
![densenet_nn](assets/densenet.jpg)

**Идея:** каждый слой получает **все предыдущие слои как вход**.  
То есть вместо обычного `x_l = F(x_{l-1})` в DenseNet:

$$
x_l = H_l([x_0, x_1, ..., x_{l-1}])
$$

где `[ \, ]` — конкатенация по каналам, а $H_l(\cdot)$ — набор операций (обычно **1×1 Conv → 3×3 Conv**).

- $x_0$ — входной тензор (например, из предыдущего блока или initial conv)
- $x_1 = H_1(x_0)$ — первый слой блока
- $x_2 = H_2([x_0, x_1])$ — второй слой получает **все предыдущие выходы**
- $x_3 = H_3([x_0, x_1, x_2])$ — третий слой получает **всё сразу**
- И так далее для всех L слоев блока.

Структура одного Dense-блока:
`вход → 1×1 Conv (сжатие) → 3×3 Conv → конкатенация с входом`


- 1×1 Conv → уменьшает число каналов → экономит вычисления  
- 3×3 Conv → извлекает новые признаки  
- Конкатенация → новые признаки добавляются к существующим, которые видны всем последующим слоям  

**Плюсы:**

- **Повторное использование признаков** → меньше параметров, чем ResNet той же глубины  
- **Хороший поток градиентов** → легче тренировать глубокие сети  
- **Уменьшение переобучения** за счет смешивания старых и новых признаков  

**Минусы:**

- Конкатенация увеличивает память  
- На inference может быть медленнее, чем ResNet


In [4]:
densenet_model = models.densenet121()
get_summary(densenet_model, False)

Всего параметров: 7,978,856
Обучаемых параметров: 7,978,856


## 4. MobileNet семейство (V1 / V2 / V3)

**Идея:** лёгкие нейросети для мобильных устройств с эффективными сверточными блоками, чтобы уменьшить количество параметров и вычисления без сильной потери точности.

#### Recap: Depthwise Separable Convolution

- **Стандартный Conv:**  
  Для входа с M каналами и выхода с N каналами:  
  $$
  \text{Params} = k^2 \cdot M \cdot N
  $$ 
  где k — размер ядра (обычно равен 3).

- **Depthwise Separable Conv** делится на два шага:

1. **Depthwise Conv:** каждому входному каналу отдельное ядро 3×3:  
$$
\text{Params} = k^2 \cdot M
$$

2. **Pointwise Conv (1×1):** объединяем каналы в выходные \(N\):  
$$
\text{Params} = M \cdot N
$$

- **Итого:**  
$$
\text{Params}_{DS} = k^2 \cdot M + M \cdot N \ll k^2 \cdot M \cdot N
$$

- **Пример:** 3×3 Conv, 32→64 канала:  
  - Стандартный Conv: 18432 параметров  
  - Depthwise + Pointwise: 2336 параметров → почти в 8 раз меньше


#### MobileNet V1 (2017)

- **Блок:** Depthwise Conv → BatchNorm → ReLU → Pointwise Conv → BatchNorm → ReLU
- **Архитектура:** Initial Conv (3×3, stride=2) → 13 DSConv блоков → Global Average Pool → FC → Softmax

![mobilenet_v1](assets/mobilenet_v1.png)

In [5]:
mobile_v1 = timm.create_model('mobilenetv1_100')
get_summary(mobile_v1)

Всего параметров: 4,231,976
Обучаемых параметров: 4,231,976


#### MobileNet V2 (2018)
![mobilenet_v2](mobilenet_v2.png)

**Идея:** улучшение MobileNet V1 через **Inverted Residual блоки** и **Linear Bottleneck**, чтобы уменьшить параметры и улучшить поток градиентов.  

**Основной блок:** Inverted Residual + Linear Bottleneck

Input → 1x1 Conv (expand) → ReLU6  
→ 3x3 Depthwise Conv → ReLU6  
→ 1x1 Conv (project) → Linear  
→ +Skip connection (stride=1)  


- **Expand layer:** увеличивает количество каналов входа → ReLU6  
- **Depthwise Conv:** отдельная свертка на каждый канал → ReLU6  
- **Project layer:** уменьшает количество каналов до исходного размера → **линейная активация** (чтобы не терять информацию)  
- **Skip connection:** добавляется только если stride=1 и размеры совпадают  


**Архитектура MobileNet V2**

- Initial Conv: 3×3 Conv, stride = 2 + BatchNorm + ReLU6
- Серия Inverted Residual Bottleneck блоков:
  - 1×1 Pointwise Conv (Expand) — увеличение числа каналов (expansion factor `t`)
  - 3×3 Depthwise Conv — spatial filtering  
    - stride = 1 → residual block  
    - stride = 2 → downsampling block
  - 1×1 Pointwise Conv (Project, linear) — проекция каналов (без нелинейности)
  - BatchNorm
  - Skip connection — только если stride = 1 и C_in = C_out
- Финальный 1×1 Conv (Pointwise) + BatchNorm + ReLU6  
- Global Average Pooling
- Fully Connected
- Softmax

**Преимущества V2**

- Меньше параметров и вычислений, чем V1 при той же точности  
- Линейный bottleneck помогает сохранять информацию через skip connections  
- Хорошо масштабируется для мобильных/встроенных приложений  


In [6]:
mobile_v2 = models.mobilenet_v2()
get_summary(mobile_v2)

Всего параметров: 3,504,872
Обучаемых параметров: 3,504,872


#### MobileNet V3 (2019)
![mobilenet_v3-small](assets/mobilenet_v3-small.png)

**MobileNet V3** — архитектура, полученная с помощью **NAS (Neural Architecture Search)**,  
объединяющая идеи:
- MobileNet V1 (DSConv)
- MobileNet V2 (Inverted Bottleneck)
- EfficientNet (scaling)
- Attention-механизмы (SE-blocks)
- Hardware-aware optimization

Авторы: Google (Howard et al., 2019)

_____

**Базовый строительный блок (MBConv + SE)**

Input
→ 1×1 Conv (Expand)
→ BN + Activation
→ 3×3 / 5×5 Depthwise Conv
→ BN + Activation
→ SE Block (Attention)
→ 1×1 Conv (Project, linear)
→ Skip connection (если stride=1 и C_in=C_out)

**Компоненты блока**

1. Inverted Bottleneck (как в V2)

2. SE Block (Squeeze-and-Excitation Attention)

![se_block](assets/se_block.jpg)

Feature Map
→ Global Avg Pool
→ FC (compression)
→ ReLU
→ FC (expansion)
→ Sigmoid
→ Channel-wise scaling

***канальное внимание (channel attention)***


**Новые активации**

1. Hard-Swish:
$$
\text{h-swish}(x) = x \cdot \frac{\text{ReLU6}(x+3)}{6}
$$

2. Hard-Sigmoid:
$$
\text{h-sigmoid}(x) = \frac{\text{ReLU6}(x+3)}{6}
$$

аппаратно-дешёвые аппроксимации sigmoid/swish


**Финальный блок MobileNet V3**

→ 1×1 Conv
→ BatchNorm
→ h-swish
→ Global Average Pooling
→ FC
→ Softmax


---

**Две версии архитектуры**

**MobileNet V3 Large**
- ориентирован на качество
- больше каналов
- глубже
- больше SE-блоков
- выше точность

**MobileNet V3 Small**
- ориентирован на мобильные устройства
- меньше каналов
- меньше блоков
- агрессивная оптимизация
- минимальная latency

_______

**Концептуальное сравнение версий**

| Версия | Парадигма |
|------|------|
| V1 | факторизация свёртки |
| V2 | геометрия embedding space |
| V3 | архитектура как система |


In [7]:
mobile_v3 = timm.create_model('mobilenetv3_small_100', pretrained=False)
get_summary(mobile_v3)

Всего параметров: 2,542,856
Обучаемых параметров: 2,542,856


In [8]:
mobile_v3 = timm.create_model('mobilenetv3_large_100', pretrained=False)
get_summary(mobile_v3)

Всего параметров: 5,483,032
Обучаемых параметров: 5,483,032


## 5. EfficientNet (2019) — Compound Scaling

![efficientnet](assets/efficientnet.png)

**Ключевая идея:**
Не масштабировать глубину, ширину и разрешение по отдельности,  
а **согласованно**:

$$
\text{depth} = \alpha^\phi,\;\;
\text{width} = \beta^\phi,\;\;
\text{resolution} = \gamma^\phi
$$

при ограничении:
$$
\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2
$$

Есть параметр масштаба:
$$
\phi = 0, 1, 2, 3, ...
$$

Он означает: "на сколько сильно мы масштабируем модель"

При увеличении $\phi$:
- сеть становится глубже
- шире
- работает с большим разрешением

одновременно, а не по отдельности

** При увеличении $\phi$ на 1 → вычисления растут примерно в 2 раза


**Архитектурная база:**
- MBConv (inverted bottleneck)
- SE attention
- Efficient scaling

**Основной блок EfficientNet — MBConv**

MBConv = Mobile Inverted Bottleneck Convolution

**Архитектура MBConv:**

Input
→ 1×1 Conv (Expand)
→ BatchNorm + Activation
→ k×k Depthwise Conv
→ BatchNorm + Activation
→ SE Block (опционально)
→ 1×1 Conv (Project, linear)
→ Skip (если stride=1 и C_in=C_out)



____
## 5. ResNeXt (2017) — Cardinality

![resnext](assets/resnext.jpg)

**Ключевая идея:**
Увеличивать **cardinality**, а не глубину или ширину

**Grouped Convolution:**

$$
\text{Conv} = \sum_{i=1}^{C} F_i(X_i)
$$

**Полный ResNeXt-блок:**

Input
→ 1×1 Conv (reduce dim)
→ 3×3 Group Conv (parallel groups + concat)
→ 1×1 Conv (channel mixing)
→ + Skip
→ ReLU

**Параметр:**
$$
\text{Cardinality} = \text{число параллельных путей}
$$

**Смысл:**
Разнообразие представлений важнее глубины

_____

## 6. ResNeSt (2020) — Split-Attention Networks

![resnest](assets/resnest.png)

**Ключевая идея:**
Внимание внутри свёрточного блока


**Split-Attention Block:**

$$
X \rightarrow \text{Split} \rightarrow \text{Transform} \rightarrow \text{Attention} \rightarrow \text{Fuse}
$$


**Формально:**

$$
Y = \sum_{i=1}^{K} a_i \cdot F_i(X), \quad \sum a_i = 1
$$

где:
- $F_i$ — параллельные группы свёрток
- $a_i$ — attention-веса
$$
a = \sigma(W_2 \cdot \text{ReLU}(W_1 \cdot GAP(\sum_{i=1}^{C} F_i(X_i)))
$$

**Смысл:**
- attention внутри Conv
- динамическая агрегация признаков
- channel-group attention

**ResNeSt Bottleneck Block**

1×1 Conv (reduce dim)
→ 3×3 Group Conv (cardinality = C)
→ Split
→ Split-Attention Module
→ 1×1 Conv (expand dim)
→ Skip connection


# 7. ConvNeXt (2022) — Modern ConvNet

![convnext](assets/convnext.png)

**Ключевая идея:**
Сделать CNN, обучаемые как Transformers


**Архитектурные принципы:**

- Large kernel conv (7×7 depthwise)
- LayerNorm вместо BatchNorm
- GELU вместо ReLU
- Inverted bottleneck
- Patchify stem
- Simplified stage design


**ConvNeXt Block:**

Input
→ 7×7 Depthwise Conv
→ LayerNorm
→ 1×1 Conv (Expand)
→ GELU
→ 1×1 Conv (Project)
→ Skip

### Stochastic Depth (DropPath)

Регуляризация, при которой во время обучения случайно “выключаются” **целые residual-блоки**, а не отдельные нейроны.  
**Цель:** предотвратить переобучение, стабилизировать обучение глубоких сетей и улучшить обобщение.

$$
y = x + b \cdot F(x), \quad b \sim \text{Bernoulli}(p)
$$

**Основная идея**

- На каждом прямом проходе каждый **residual-блок** имеет вероятность `p` быть **включённым**.
- С вероятностью `1-p` блок **полностью пропускается** (остается только skip-connection).
- Выключенные блоки:
  - не участвуют в вычислении выхода,
  - не получают градиенты.
- Во время инференса все блоки активны, но необходима компенсация масштаба.


Компенсация во время обучения:

$$
\tilde{F}(x) = \frac{F(x) \cdot b}{p}
$$

Полная формула:

$$
y = x + \frac{b}{p} \cdot F(x)
$$

**Структурный смысл**

- Сеть обучается как **ансамбль моделей разной глубины**
- На каждом шаге обучения:
  - иногда сеть глубокая
  - иногда “мелкая”
  - иногда средней глубины
- Это улучшает:
  - обобщение (generalization)
  - устойчивость к переобучению
  - стабильность градиентов
_____

**Эволюционная логика**

| Архитектура | Оптимизация |
|------|------|
| ResNet | градиенты |
| ResNeXt | параллелизм |
| ResNeSt | внимание |
| EfficientNet | масштабирование |
| ConvNeXt | обучение как у Transformers |




In [10]:
# ResNet
print("ResNet")
get_summary(resnet_model, False)

# EfficientNet
efficientnet = models.efficientnet_b0()
print("\n\nEfficientNet")
get_summary(efficientnet, False)

# ResNeSt
resnest = resnest50(pretrained=False)
print("\n\nResNeSt")
get_summary(resnest, False)

# ResNeXt
resnext = models.resnext50_32x4d()
print("\n\nResNeXt")
get_summary(resnext, False)

# ConvNeXt
convnext = models.convnext_tiny()
print("\n\nConvNeSt")
get_summary(convnext, False)


ResNet
Всего параметров: 25,557,032
Обучаемых параметров: 25,557,032


EfficientNet
Всего параметров: 5,288,548
Обучаемых параметров: 5,288,548


ResNeSt
Всего параметров: 27,483,240
Обучаемых параметров: 27,483,240


ResNeXt
Всего параметров: 25,028,904
Обучаемых параметров: 25,028,904


ConvNeSt
Всего параметров: 28,589,128
Обучаемых параметров: 28,589,128
